In [5]:
# Set GPU
import os
os.environ["CUDA_VISIBLE_DEVICES"]="2"

In [6]:
from torch.utils.data import Dataset
import os, random, time
import cv2 as cv2
from matplotlib import pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np

import torch
import torchvision
from torchvision import transforms
from torchvision import io
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import FasterRCNN, fasterrcnn_resnet50_fpn_v2, fasterrcnn_resnet50_fpn, fasterrcnn_mobilenet_v3_large_320_fpn
from torchvision.models.detection.rpn import AnchorGenerator
import torchvision.transforms.functional as F_vision
from torchvision.utils import make_grid
from torch.utils.data import DataLoader, Dataset
from torch.utils.data.sampler import SequentialSampler
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone

from tqdm.notebook import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import albumentations.pytorch
import pandas as pd
from PIL import Image
import imageio
import math
from pathlib import Path
from os.path import isfile, join
import xml.etree.ElementTree as ET
import pandas as pd
import matplotlib.pyplot as plt
from torchmetrics.detection import MeanAveragePrecision
import json
import glob
import re

from config import *
from helpers import *

In [7]:
# Input
folder_models = 'models'
folder_data = 'data'

In [10]:
# Loop over model files
all_results = {}
for file_model in glob.glob(f'{folder_models}/*/modelo*.bin'):
    print()
    print(file_model)
    
    # Prepare model
    model, device, bbox_params = prepare_model(file_model, verbose=False)
    
    # Predict val set
    all_bboxes_true, all_bboxes_pred, all_scores = predict_val(folder_data, bbox_params, model, device)
    
    # Evaluate
    results = evaluate(all_bboxes_true, all_bboxes_pred, all_scores)
        
    # Append
    key = file_model.split('/')[-2]
    all_results[key] = results


models/0070_fasterrcnn_resnet50_fpn_th50/modelo.bin


KeyboardInterrupt: 

In [53]:
# Backbone Confidence Threshold Refinement Step deblurGAN mAP mAR
rows = []
for key in keys:
    row = {}
    # row['key'] = key
    match = re.search(r"resnet[0-9]+", key)
    if match:
        backbone_name = match.group()
        row['backbone'] = (backbone_name)

    row['Confidence Threshold'] = 0.5
    row['Refinement Step'] = 'Yes'
    if 'deblurGAN' in key:
        row['deblurGAN'] = 'Yes'
    else:
        row['deblurGAN'] = 'No'

    row['mAP'] = all_results[key]['map']

    rows.append(row)

pd.DataFrame(rows)

,backbone,Confidence Threshold,Refinement Step,deblurGAN,mAP
0,resnet18,0.5,Yes,No,0.902809
1,resnet18,0.5,Yes,Yes,0.826668
2,resnet50,0.5,Yes,No,0.927313
3,resnet50,0.5,Yes,Yes,0.647665
4,resnet152,0.5,Yes,No,0.872988
5,resnet152,0.5,Yes,Yes,0.815298


In [7]:
# Write results to file
with open("results.json", "w") as f:
    json.dump(all_results, f, indent=2)